# MutualInfoSelector

Scores each feature by its mutual information with the target: how much
knowing the feature reduces uncertainty about the class.

It is estimated from nearest neighbours, so it detects any kind of
dependence, not only a shift in the mean.

- **Cost:** much slower than ANOVA F; it runs a neighbour search per feature.
- **Strength:** catches curved and non-monotone relationships.
- **Weakness:** the estimate is noisy on small samples, and it depends on
  `random_state` and `n_neighbors`.
- **Requires** `n_features`.

In [1]:
import numpy as np

rng = np.random.default_rng(0)
from featsel.selectors import ANOVAFSelector, MutualInfoSelector

# Class depends on |x0| (curved); x1 is a plain mean shift; the rest is noise.
n = 1000
x0 = rng.normal(size=n)
y = (np.abs(x0) > 1).astype(int)
x1 = rng.normal(size=n) + 0.8 * y
X = np.column_stack([x0, x1, rng.normal(size=(n, 48))])

## Mutual information against ANOVA F on the same data

In [2]:
mi = MutualInfoSelector(n_features=2, random_state=0).fit(X, y)
anova = ANOVAFSelector(n_features=2).fit(X, y)

print('mutual information picks:', sorted(mi.get_support(indices=True).tolist()))
print('ANOVA F picks           :', sorted(anova.get_support(indices=True).tolist()))

mutual information picks: [0, 1]
ANOVA F picks           : [1, 33]


Both find `x1`, the mean shift. Only mutual information also finds `x0`,
whose classes share the same mean.

## The seed

The seed only adds tiny jitter to the features to break ties between equal
values. Here the scores do not move at the third decimal.

In [3]:
for seed in (0, 1, 2):
    scores = MutualInfoSelector(n_features=2, random_state=seed).fit(X, y).feature_importances_
    print(f'seed {seed}: x0 {scores[0]:.3f}  x1 {scores[1]:.3f}  best noise {scores[2:].max():.3f}')

seed 0: x0 0.620  x1 0.075  best noise 0.041


seed 1: x0 0.620  x1 0.075  best noise 0.041


seed 2: x0 0.620  x1 0.075  best noise 0.041
